In [1]:
import os
import sys
import json
import numpy as np
import pandas as pd

# Memastikan driver dan worker Spark menggunakan interpreter Python yang sama
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

np.random.seed(123)

# Sumber 1: Data cabang (JSON)
cabang_list = ["Kopi Nusantara Magelang", "Kopi Nusantara Yogyakarta", "Kopi Nusantara Semarang"]
data_cabang = [
    {"cabang_id": i + 1, "nama_cabang": c, "kota": c.split()[-1],
     "kepala_barista": np.random.choice(["Wulan", "Reza", "Nadia"])}
    for i, c in enumerate(cabang_list)
]
with open("tugas7_cabang.json", "w") as f:
    for c in data_cabang:
        f.write(json.dumps(c) + "\n")

# Sumber 2: Data menu (CSV)
menu_list = [
    {"menu_id": i, "nama_menu": f"Menu-{i}",
     "kategori_menu": np.random.choice(["Kopi", "Non-Kopi", "Makanan Ringan"]),
     "harga": int(np.random.choice([18000, 22000, 25000, 30000, 35000]))}
    for i in range(1, 16)
]
pd.DataFrame(menu_list).to_csv("tugas7_menu.csv", index=False)

# Sumber 3: Data transaksi (CSV) — data utama
n = 4000
data_transaksi_tugas7 = pd.DataFrame({
    "trx_id": [f"KN{i}" for i in range(n)],
    "cabang_id": np.random.randint(1, 4, size=n),
    "menu_id": np.random.randint(1, 16, size=n),
    "qty": np.random.randint(1, 5, size=n),
})
data_transaksi_tugas7.to_csv("tugas7_transaksi.csv", index=False)

!hdfs dfs -mkdir -p /user/mahasiswa/tugas7/raw
!hdfs dfs -put -f tugas7_transaksi.csv /user/mahasiswa/tugas7/raw/
!hdfs dfs -put -f tugas7_cabang.json /user/mahasiswa/tugas7/raw/
!hdfs dfs -put -f tugas7_menu.csv /user/mahasiswa/tugas7/raw/

print("Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.")
print(f"- Cabang: {len(data_cabang)} | Menu: {len(menu_list)} | Transaksi: {n}")

Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.
- Cabang: 3 | Menu: 15 | Transaksi: 4000


---
## Inisialisasi SparkSession

Membuat sesi Spark (`SparkSession`) dengan konfigurasi mode lokal paralel (`local[*]`) untuk memproses beban kerja analisis data terdistribusi.

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, row_number, round as spark_round
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Tugas7_2505060072_Muhammad_Raka_Mushaddaq") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("SparkSession berhasil diinisialisasi.")
print("Versi Apache Spark :", spark.version)
print("Aplikasi Spark     :", spark.sparkContext.appName)

SparkSession berhasil diinisialisasi.
Versi Apache Spark : 3.5.9
Aplikasi Spark     : Tugas7_2505060072_Muhammad_Raka_Mushaddaq


---
## Bagian A. EXTRACT

**Instruksi:**
> Baca ketiga sumber data (`tugas7_transaksi.csv`, `tugas7_cabang.json`, `tugas7_menu.csv`) **dari HDFS** (bukan disk lokal) menjadi tiga Spark DataFrame terpisah. Tampilkan jumlah baris dan `printSchema()` masing-masing.

In [5]:
# 1. Membaca Data Transaksi dari HDFS (CSV)
df_transaksi_kn = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_transaksi.csv",
    header=True, inferSchema=True
)

# 2. Membaca Data Cabang dari HDFS (JSON Lines)
df_cabang_kn = spark.read.json(
    "hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_cabang.json"
)

# 3. Membaca Data Menu dari HDFS (CSV)
df_menu_kn = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_menu.csv",
    header=True, inferSchema=True
)

print("=" * 60)
print("HASIL TAHAP EXTRACT (DIBACA LANGSUNG DARI HDFS)")
print("=" * 60)
print(f"1. df_transaksi_kn : {df_transaksi_kn.count():,} baris")
print(f"2. df_cabang_kn    : {df_cabang_kn.count():,} baris")
print(f"3. df_menu_kn      : {df_menu_kn.count():,} baris")

print("\n--- Skema df_transaksi_kn ---")
df_transaksi_kn.printSchema()

print("\n--- Skema df_cabang_kn ---")
df_cabang_kn.printSchema()

print("\n--- Skema df_menu_kn ---")
df_menu_kn.printSchema()

print("\nCuplikan 3 baris data transaksi:")
df_transaksi_kn.show(3)

HASIL TAHAP EXTRACT (DIBACA LANGSUNG DARI HDFS)
1. df_transaksi_kn : 4,000 baris
2. df_cabang_kn    : 3 baris
3. df_menu_kn      : 15 baris

--- Skema df_transaksi_kn ---
root
 |-- trx_id: string (nullable = true)
 |-- cabang_id: integer (nullable = true)
 |-- menu_id: integer (nullable = true)
 |-- qty: integer (nullable = true)


--- Skema df_cabang_kn ---
root
 |-- cabang_id: long (nullable = true)
 |-- kepala_barista: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- nama_cabang: string (nullable = true)


--- Skema df_menu_kn ---
root
 |-- menu_id: integer (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- kategori_menu: string (nullable = true)
 |-- harga: integer (nullable = true)


Cuplikan 3 baris data transaksi:
+------+---------+-------+---+
|trx_id|cabang_id|menu_id|qty|
+------+---------+-------+---+
|   KN0|        1|     14|  4|
|   KN1|        1|     11|  4|
|   KN2|        1|     11|  3|
+------+---------+-------+---+
only showing top 3 row

---
## Bagian B. TRANSFORM — Join & Kolom Turunan (Bobot 20%)

**Instruksi:**
> Gabungkan ketiga tabel (`df_transaksi_kn`, `df_cabang_kn`, `df_menu_kn`), tambahkan kolom `total_penjualan` (`qty x harga`).

In [6]:
# Bagian B: TRANSFORM - Join Ketiga Tabel & Penambahan Kolom total_penjualan
# Langkah 1: Melakukan penggabungan transaksi ke cabang (pada cabang_id) dan ke menu (pada menu_id)
df_gabungan_kn = df_transaksi_kn.join(df_cabang_kn, on="cabang_id", how="inner") \
                                .join(df_menu_kn, on="menu_id", how="inner")

# Langkah 2: Menambahkan kolom kalkulasi total_penjualan = qty * harga
df_gabungan_kn = df_gabungan_kn.withColumn("total_penjualan", col("qty") * col("harga"))

print("=" * 60)
print("HASIL TRANSFORMASI: PENGGABUNGAN DATA & TOTAL PENJUALAN")
print("=" * 60)
print(f"Total baris data gabungan : {df_gabungan_kn.count():,} baris")
print("\nSkema Data Gabungan:")
df_gabungan_kn.printSchema()

print("\nCuplikan 5 baris pertama data hasil gabungan:")
df_gabungan_kn.select("trx_id", "nama_cabang", "kota", "kepala_barista", "nama_menu", "kategori_menu", "harga", "qty", "total_penjualan").show(5, truncate=False)

HASIL TRANSFORMASI: PENGGABUNGAN DATA & TOTAL PENJUALAN
Total baris data gabungan : 4,000 baris

Skema Data Gabungan:
root
 |-- menu_id: integer (nullable = true)
 |-- cabang_id: integer (nullable = true)
 |-- trx_id: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- kepala_barista: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- nama_cabang: string (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- kategori_menu: string (nullable = true)
 |-- harga: integer (nullable = true)
 |-- total_penjualan: integer (nullable = true)


Cuplikan 5 baris pertama data hasil gabungan:
+------+-------------------------+----------+--------------+---------+--------------+-----+---+---------------+
|trx_id|nama_cabang              |kota      |kepala_barista|nama_menu|kategori_menu |harga|qty|total_penjualan|
+------+-------------------------+----------+--------------+---------+--------------+-----+---+---------------+
|KN0   |Kopi Nusantara Magelang  |Magela

---
## Bagian C. TRANSFORM — Window Function (Bobot 20%)

**Instruksi:**
> Tentukan **top-2 menu terlaris** (berdasarkan `total_penjualan`) di **setiap cabang**, gunakan `row_number()`.

In [8]:
# Bagian C: TRANSFORM - Window Function Top-2 Menu Terlaris per Cabang
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, sum as spark_sum, col

# Langkah 1: Meringkas akumulasi penjualan dan total cup per kombinasi cabang dan menu
ringkasan_cabang_menu = df_gabungan_kn.groupBy("nama_cabang", "nama_menu", "kategori_menu", "harga").agg(
    spark_sum("total_penjualan").alias("total_penjualan"),
    spark_sum("qty").alias("total_cup_terjual")
)

# Langkah 2: Menentukan spesifikasi Window yang mempartisi per cabang dan mengurutkan omset terbesar ke terkecil
window_cabang = Window.partitionBy("nama_cabang").orderBy(col("total_penjualan").desc())

# Langkah 3: Menerapkan fungsi analitis row_number() dan memfilter hanya peringkat 1 dan 2
top2_menu_per_cabang = ringkasan_cabang_menu.withColumn("peringkat", row_number().over(window_cabang)) \
    .filter(col("peringkat") <= 2) \
    .orderBy("nama_cabang", "peringkat")

print("=" * 60)
print("TOP-2 MENU TERLARIS BERDASARKAN TOTAL PENJUALAN DI SETIAP CABANG")
print("=" * 60)
top2_menu_per_cabang.show(truncate=False)

TOP-2 MENU TERLARIS BERDASARKAN TOTAL PENJUALAN DI SETIAP CABANG
+-------------------------+---------+--------------+-----+---------------+-----------------+---------+
|nama_cabang              |nama_menu|kategori_menu |harga|total_penjualan|total_cup_terjual|peringkat|
+-------------------------+---------+--------------+-----+---------------+-----------------+---------+
|Kopi Nusantara Magelang  |Menu-1   |Makanan Ringan|35000|8505000        |243              |1        |
|Kopi Nusantara Magelang  |Menu-12  |Makanan Ringan|35000|8085000        |231              |2        |
|Kopi Nusantara Semarang  |Menu-13  |Makanan Ringan|35000|8750000        |250              |1        |
|Kopi Nusantara Semarang  |Menu-9   |Makanan Ringan|35000|8715000        |249              |2        |
|Kopi Nusantara Yogyakarta|Menu-12  |Makanan Ringan|35000|9275000        |265              |1        |
|Kopi Nusantara Yogyakarta|Menu-15  |Non-Kopi      |30000|8160000        |272              |2        |
+-------

---
## Bagian D. TRANSFORM — Spark SQL (Bobot 20%)

**Instruksi:**
> Menggunakan **Spark SQL murni**, tampilkan total penjualan per `kepala_barista`, diurutkan dari tertinggi.

In [11]:
# Bagian D: TRANSFORM - Analisis Kontribusi Kepala Barista Menggunakan Spark SQL Murni
# Mendaftarkan DataFrame gabungan sebagai Temporary View
df_gabungan_kn.createOrReplaceTempView("transaksi_kopi_nusantara")

# Menjalankan kueri SQL murni untuk mengagregasi total penjualan per kepala barista
penjualan_per_barista = spark.sql('''
    SELECT 
        kepala_barista,
        COUNT(trx_id) AS jumlah_transaksi,
        SUM(qty) AS total_cup_terjual,
        SUM(total_penjualan) AS total_penjualan,
        ROUND(AVG(total_penjualan), 2) AS rata_rata_penjualan_per_trx
    FROM transaksi_kopi_nusantara
    GROUP BY kepala_barista
    ORDER BY total_penjualan DESC
''')

print("=" * 60)
print("KONTRIBUSI PENJUALAN PER KEPALA BARISTA (SPARK SQL MURNI)")
print("=" * 60)
penjualan_per_barista.show(truncate=False)

KONTRIBUSI PENJUALAN PER KEPALA BARISTA (SPARK SQL MURNI)
+--------------+----------------+-----------------+---------------+---------------------------+
|kepala_barista|jumlah_transaksi|total_cup_terjual|total_penjualan|rata_rata_penjualan_per_trx|
+--------------+----------------+-----------------+---------------+---------------------------+
|Nadia         |2695            |6734             |177680000      |65929.5                    |
|Reza          |1305            |3205             |84645000       |64862.07                   |
+--------------+----------------+-----------------+---------------+---------------------------+



---
## Bagian E. LOAD (Bobot 15%)

**Instruksi:**
> Simpan data gabungan (hasil bagian B) ke HDFS sebagai Parquet, path `/user/[username]/tugas7/processed/laporan`, dipartisi berdasarkan `kategori_menu`. Verifikasi dengan membaca kembali dan menghitung jumlah baris.

In [15]:
# Bagian E: LOAD - Menyimpan Data Gabungan ke HDFS Parquet Terpartisi & Verifikasi
# Menentukan path target penyimpanan di HDFS
hdfs_laporan_path = "hdfs://localhost:9000/user/mahasiswa/tugas7/processed/laporan"

# Menuliskan data gabungan ke HDFS berformat Parquet dipartisi berdasarkan 'kategori_menu'
df_gabungan_kn.write.mode("overwrite").partitionBy("kategori_menu").parquet(hdfs_laporan_path)
print(f"Data laporan berhasil disimpan ke HDFS: {hdfs_laporan_path}\n")

# Verifikasi struktur direktori partisi di HDFS
print("=== Struktur Direktori Partisi di HDFS ===")
!hdfs dfs -ls /user/mahasiswa/tugas7/processed/laporan

# Membaca kembali data Parquet dari HDFS untuk verifikasi keutuhan baris
df_verifikasi_laporan = spark.read.parquet(hdfs_laporan_path)

print("\n" + "=" * 60)
print("VERIFIKASI MEMBACA KEMBALI LAPORAN DARI HDFS")
print("=" * 60)
print(f"Jumlah baris awal (Bagian B)   : {df_gabungan_kn.count():,} baris")
print(f"Jumlah baris terbaca dari HDFS : {df_verifikasi_laporan.count():,} baris")
print(f"Status Integritas Data         : {'VALID (SAMA PERSIS)' if df_gabungan_kn.count() == df_verifikasi_laporan.count() else 'TIDAK SESUAI'}")

print("\nSkema Data Parquet Terbaca:")
df_verifikasi_laporan.printSchema()

Data laporan berhasil disimpan ke HDFS: hdfs://localhost:9000/user/mahasiswa/tugas7/processed/laporan

=== Struktur Direktori Partisi di HDFS ===
Found 4 items
-rw-r--r--   3 raka supergroup          0 2026-10-04 11:12 /user/mahasiswa/tugas7/processed/laporan/_SUCCESS
drwxr-xr-x   - raka supergroup          0 2026-10-04 11:12 /user/mahasiswa/tugas7/processed/laporan/kategori_menu=Kopi
drwxr-xr-x   - raka supergroup          0 2026-10-04 11:12 /user/mahasiswa/tugas7/processed/laporan/kategori_menu=Makanan Ringan
drwxr-xr-x   - raka supergroup          0 2026-10-04 11:12 /user/mahasiswa/tugas7/processed/laporan/kategori_menu=Non-Kopi

VERIFIKASI MEMBACA KEMBALI LAPORAN DARI HDFS
Jumlah baris awal (Bagian B)   : 4,000 baris
Jumlah baris terbaca dari HDFS : 4,000 baris
Status Integritas Data         : VALID (SAMA PERSIS)

Skema Data Parquet Terbaca:
root
 |-- menu_id: integer (nullable = true)
 |-- cabang_id: integer (nullable = true)
 |-- trx_id: string (nullable = true)
 |-- qty: integer

---
## Bagian F. Ringkasan Eksekutif & Rekomendasi Bisnis (Bobot 15%)

**Instruksi:**
> Buat tabel ringkasan (per cabang: total penjualan, jumlah transaksi, rata-rata nilai transaksi), lalu tulis pada markdown cell (**minimal 120 kata**): cabang mana yang berkinerja terbaik, menu apa yang sebaiknya dipromosikan lebih gencar di cabang dengan kinerja terlemah (berdasarkan temuan bagian C), dan alasannya.

In [17]:
# Bagian F: Ringkasan Eksekutif Kinerja Finansial Cabang Kopi Nusantara
from pyspark.sql.functions import avg, count, round as spark_round, sum as spark_sum

# Menyusun ringkasan eksekutif dari data laporan yang dibaca dari HDFS
ringkasan_eksekutif_kn = df_verifikasi_laporan.groupBy("nama_cabang", "kota", "kepala_barista").agg(
    spark_sum("total_penjualan").alias("total_penjualan"),
    count("trx_id").alias("jumlah_transaksi"),
    spark_round(avg("total_penjualan"), 2).alias("rata_rata_nilai_transaksi")
).orderBy(col("total_penjualan").desc())

print("=" * 70)
print("RINGKASAN EKSEKUTIF KINERJA CABANG KOPI NUSANTARA")
print("=" * 70)
ringkasan_eksekutif_kn.show(truncate=False)

RINGKASAN EKSEKUTIF KINERJA CABANG KOPI NUSANTARA
+-------------------------+----------+--------------+---------------+----------------+-------------------------+
|nama_cabang              |kota      |kepala_barista|total_penjualan|jumlah_transaksi|rata_rata_nilai_transaksi|
+-------------------------+----------+--------------+---------------+----------------+-------------------------+
|Kopi Nusantara Semarang  |Semarang  |Nadia         |90542000       |1393            |64997.85                 |
|Kopi Nusantara Magelang  |Magelang  |Nadia         |87138000       |1302            |66926.27                 |
|Kopi Nusantara Yogyakarta|Yogyakarta|Reza          |84645000       |1305            |64862.07                 |
+-------------------------+----------+--------------+---------------+----------------+-------------------------+



In [18]:
# Menutup sesi Spark setelah seluruh pipeline selesai dieksekusi
spark.stop()
print("SparkSession Tugas Mandiri berhasil ditutup secara aman.")

SparkSession Tugas Mandiri berhasil ditutup secara aman.
